# Simplified Stress Test — Greek Systemic Banks (one year, FY2024 base)

A one-year shock applied to the FY2024 figures of each listed group, to see how much profit and capital it would take away.

| Shock | Size | Rationale |
|-------|------|-----------|
| Cost of risk | +200 bps on the loan book | Order of magnitude of the first-year credit losses in EBA adverse scenarios |
| Loan volume | −15% | Credit contraction; earning assets fall by the same amount |
| NIM | −50 bps | Margin compression from rate cuts and deposit repricing |
| Operating costs | Flat | No cost cutting assumed |
| Tax | 29% on the change in profit | Greek corporate tax rate for credit institutions |
| Dividends | None in the stress year | Banks would suspend payouts under stress |
| RWAs | Unchanged | Conservative: a smaller loan book would lower RWAs |

**CET1 at the end of the stress year** = (CET1 capital at end-2024 + stressed net profit) / RWAs, with CET1 capital = reported CET1 ratio × reported RWAs.
**Loss-absorption capacity** = extra cost of risk (bps of loans) that would bring FY2024 profit before tax to zero.

**Context, the official EBA 2023 stress test** (three-year adverse scenario, static balance sheet at 31.12.2022, fully loaded CET1 at end-2025, as reported by each bank): Alpha 11.9% → 8.9%, Eurobank 14.4% → 12.2%, NBG 15.9% → 14.5%, Piraeus 11.5% → 9.1%. Capital has risen strongly since that starting point.

> **Disclaimer:** a simplified, static, one-year exercise for illustration. It does not model sovereign bond losses, funding shocks, operational risk or RWA migration.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA = Path('../02_Banking_Sector_Dashboard/data')

kpis = pd.read_csv(DATA / 'processed' / 'kpis_final.csv')
eba = pd.read_csv(DATA / 'manual' / 'eba_stress_test_2023.csv').set_index('bank')
df24 = kpis[kpis['year'] == 2024].set_index('bank').loc[BANKS]

df24['nim_dec'] = df24['nii'] / df24['avg_total_assets']
df24['cor_dec'] = -df24['impairment'] / df24['avg_loans']
df24['cet1_capital'] = df24['cet1'] / 100 * df24['rwa']

print('FY2024 base:')
print(df24[['nii', 'pbt', 'net_profit', 'loans', 'cor_dec', 'cet1', 'rwa', 'cet1_capital']].round(4).to_string())

FY2024 base:
                 nii     pbt  net_profit    loans  cor_dec   cet1      rwa  cet1_capital
bank                                                                                    
Eurobank      2507.0  1872.0      1448.0  50953.0   0.0066  16.80  49977.0     8396.1360
Alpha Bank    1645.0   816.0       654.0  39050.0   0.0096  16.30  30300.0     4938.9000
Piraeus Bank  2088.0  1436.0      1066.0  40735.0   0.0046  14.47  34098.0     4933.9806
NBG           2356.0  1517.0      1158.0  36139.0   0.0051  18.30  37400.0     6844.2000


In [2]:
# ── Stress parameters ─────────────────────────────────────────────────────────
STRESS = {'cor_shock_bp': 200, 'loan_shock_pct': -0.15, 'nim_shock_bp': -50}
TAX_RATE = 0.29

rows = []
for bank in BANKS:
    r = df24.loc[bank]
    loans_adv  = r['loans'] * (1 + STRESS['loan_shock_pct'])
    assets_adv = r['avg_total_assets'] + r['loans'] * STRESS['loan_shock_pct']   # earning assets shrink with loans
    nim_adv    = r['nim_dec'] + STRESS['nim_shock_bp'] / 10_000
    cor_adv    = r['cor_dec'] + STRESS['cor_shock_bp'] / 10_000

    nii_adv  = nim_adv * assets_adv
    imp_adv  = cor_adv * loans_adv                     # positive = charge
    imp_base = -r['impairment']
    pbt_adv  = r['pbt'] + (nii_adv - r['nii']) - (imp_adv - imp_base)
    np_adv   = r['net_profit'] + (pbt_adv - r['pbt']) * (1 - TAX_RATE)

    cet1_end_stress = (r['cet1_capital'] + np_adv) / r['rwa'] * 100
    rows.append({
        'bank': bank,
        'net_profit_base': r['net_profit'], 'net_profit_adv': np_adv,
        'pbt_base': r['pbt'], 'pbt_adv': pbt_adv,
        'cet1_start': r['cet1'], 'cet1_end_stress': cet1_end_stress,
        'profit_hit_cet1_pp': (np_adv - r['net_profit']) / r['rwa'] * 100,
        'loss_absorption_bp': r['pbt'] / r['loans'] * 10_000,
        'eba_2023_adverse_2025_fl': eba.loc[bank, 'cet1_fl_adverse_2025'],
    })

st = pd.DataFrame(rows).set_index('bank')
print('Stress results (€ million, %):')
print(st.round(1).to_string())

Stress results (€ million, %):
              net_profit_base  net_profit_adv  pbt_base  pbt_adv  cet1_start  cet1_end_stress  profit_hit_cet1_pp  loss_absorption_bp  eba_2023_adverse_2025_fl
bank                                                                                                                                                           
Eurobank               1448.0           402.3    1872.0    399.1        16.8             17.6                -2.1               367.4                      12.2
Alpha Bank              654.0          -117.7     816.0   -270.9        16.3             15.9                -2.5               209.0                       8.9
Piraeus Bank           1066.0           217.4    1436.0    240.7        14.5             15.1                -2.5               352.5                       9.1
NBG                    1158.0           370.5    1517.0    407.9        18.3             19.3                -2.1               419.8                      14.5


In [3]:
# ── Chart 1: Net profit and CET1, base vs stress ──────────────────────────────
fig1 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=['<b>Net profit: FY2024 vs stress year (€m)</b>',
                    '<b>CET1: end-2024 vs end of stress year (%)</b>'])

fig1.add_trace(go.Bar(x=BANKS, y=st['net_profit_base'], name='FY2024 actual',
    marker_color=[COLORS[b] for b in BANKS], text=[f'{v:,.0f}' for v in st['net_profit_base']],
    textposition='outside'), row=1, col=1)
fig1.add_trace(go.Bar(x=BANKS, y=st['net_profit_adv'], name='Stress year', marker_color='#888888',
    text=[f'{v:,.0f}' for v in st['net_profit_adv']], textposition='outside'), row=1, col=1)
fig1.add_hline(y=0, line_color='rgba(255,255,255,0.4)', row=1, col=1)

fig1.add_trace(go.Bar(x=BANKS, y=st['cet1_start'], name='CET1 end-2024 (reported)', showlegend=False,
    marker_color=[COLORS[b] for b in BANKS], text=[f'{v:.1f}%' for v in st['cet1_start']],
    textposition='outside'), row=1, col=2)
fig1.add_trace(go.Bar(x=BANKS, y=st['cet1_end_stress'], name='CET1 end of stress year', showlegend=False,
    marker_color='#888888', text=[f'{v:.1f}%' for v in st['cet1_end_stress']],
    textposition='outside'), row=1, col=2)

fig1.update_layout(
    title_text=(f'<b>One-year stress</b> | CoR +{STRESS["cor_shock_bp"]}bps | Loans {STRESS["loan_shock_pct"]*100:.0f}% '
                f'| NIM {STRESS["nim_shock_bp"]}bps | no dividends'),
    title_font_size=15, template='plotly_dark', height=440, barmode='group',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.show()

In [4]:
# ── Chart 2: Loss-absorption capacity and the official EBA 2023 result ────────
fig2 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=['<b>Extra cost of risk before a pre-tax loss (bps, FY2024)</b>',
                    '<b>EBA 2023 stress test: fully loaded CET1 (%)</b>'])

order = st['loss_absorption_bp'].sort_values(ascending=False).index
fig2.add_trace(go.Bar(x=list(order), y=st.loc[order, 'loss_absorption_bp'],
    marker_color=[COLORS[b] for b in order], text=[f'{v:.0f}' for v in st.loc[order, 'loss_absorption_bp']],
    textposition='outside', showlegend=False), row=1, col=1)

fig2.add_trace(go.Bar(x=BANKS, y=eba.loc[BANKS, 'cet1_fl_start_2022'], name='Start (Dec 2022)',
    marker_color=[COLORS[b] for b in BANKS], text=[f'{v:.1f}%' for v in eba.loc[BANKS, 'cet1_fl_start_2022']],
    textposition='outside'), row=1, col=2)
fig2.add_trace(go.Bar(x=BANKS, y=eba.loc[BANKS, 'cet1_fl_adverse_2025'], name='Adverse, end-2025',
    marker_color='#888888', text=[f'{v:.1f}%' for v in eba.loc[BANKS, 'cet1_fl_adverse_2025']],
    textposition='outside'), row=1, col=2)

fig2.update_layout(template='plotly_dark', height=420, barmode='group',
    legend=dict(orientation='h', y=-0.18, x=0.75, xanchor='center'),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117')
fig2.show()

In [5]:
# ── Final validation and summary ──────────────────────────────────────────────
assert len(st) == 4
assert (st['net_profit_adv'] < st['net_profit_base']).all(), 'Stress must reduce profit'
assert (st['pbt_adv'] < st['pbt_base']).all()

print('Summary')
for bank, r in st.sort_values('cet1_end_stress', ascending=False).iterrows():
    print(f"  {bank:13s} stressed net profit €{r['net_profit_adv']:>7,.0f}m | CET1 {r['cet1_start']:.1f}% → "
          f"{r['cet1_end_stress']:.1f}% at end of stress year | absorbs +{r['loss_absorption_bp']:.0f} bps CoR before a pre-tax loss")
weakest = st['loss_absorption_bp'].idxmin()
print(f"\nLowest loss-absorption capacity: {weakest} (+{st.loc[weakest,'loss_absorption_bp']:.0f} bps).")
print('In the official EBA 2023 exercise the lowest end-2025 adverse CET1 were Alpha (8.9%) and Piraeus (9.1%).')
print('\nAll checks passed.')

Summary
  NBG           stressed net profit €    371m | CET1 18.3% → 19.3% at end of stress year | absorbs +420 bps CoR before a pre-tax loss
  Eurobank      stressed net profit €    402m | CET1 16.8% → 17.6% at end of stress year | absorbs +367 bps CoR before a pre-tax loss
  Alpha Bank    stressed net profit €   -118m | CET1 16.3% → 15.9% at end of stress year | absorbs +209 bps CoR before a pre-tax loss
  Piraeus Bank  stressed net profit €    217m | CET1 14.5% → 15.1% at end of stress year | absorbs +353 bps CoR before a pre-tax loss

Lowest loss-absorption capacity: Alpha Bank (+209 bps).
In the official EBA 2023 exercise the lowest end-2025 adverse CET1 were Alpha (8.9%) and Piraeus (9.1%).

All checks passed.
